# Chapter 7, Exercise 3: Inter-annotator agreement for dialect labels

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 7, Exercise 3.** Three annotators label 20 short clips for dialect from four classes. Using Python and a standard library (for example statsmodels' `fleiss_kappa` for several raters, or scikit-learn's `cohen_kappa_score` for the two-rater case), compute the inter-annotator agreement: provide a small sample of labels, report the score, say what value would indicate usable agreement, and list what a one-page data card for the resulting set must contain.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


In [1]:
!pip install -q statsmodels scikit-learn pandas

## 1. A small sample of labels

Twenty clips, three annotators (A1, A2, A3), four dialect classes: MSA, Egyptian, Gulf, Levantine. The labels below are **illustrative** (made up to show the computation); replace them with your own. Disagreements are placed mostly between neighbouring varieties (Gulf/Levantine) and on MSA-versus-dialect boundaries, the patterns Chapter 8 predicts.

In [2]:
import pandas as pd, numpy as np
CLASSES = ["MSA", "Egyptian", "Gulf", "Levantine"]
labels = pd.DataFrame({
    "clip": [f"clip_{i:02d}" for i in range(1, 21)],
    "A1": ["MSA","MSA","Egyptian","Egyptian","Gulf","Gulf","Levantine","Levantine","Gulf","MSA",
           "Egyptian","Levantine","Gulf","MSA","Egyptian","Gulf","Levantine","MSA","Egyptian","Gulf"],
    "A2": ["MSA","MSA","Egyptian","Egyptian","Gulf","Levantine","Levantine","Levantine","Gulf","MSA",
           "Egyptian","Gulf","Gulf","Egyptian","Egyptian","Gulf","Levantine","MSA","Egyptian","Levantine"],
    "A3": ["MSA","Egyptian","Egyptian","Egyptian","Gulf","Gulf","Levantine","Gulf","Gulf","MSA",
           "Egyptian","Levantine","Gulf","MSA","Egyptian","Gulf","Levantine","MSA","MSA","Gulf"],
})
labels

,clip,A1,A2,A3
0,clip_01,MSA,MSA,MSA
1,clip_02,MSA,MSA,Egyptian
2,clip_03,Egyptian,Egyptian,Egyptian
3,clip_04,Egyptian,Egyptian,Egyptian
4,clip_05,Gulf,Gulf,Gulf
5,clip_06,Gulf,Levantine,Gulf
6,clip_07,Levantine,Levantine,Levantine
7,clip_08,Levantine,Levantine,Gulf
8,clip_09,Gulf,Gulf,Gulf
9,clip_10,MSA,MSA,MSA


## 2. Fleiss' kappa for three raters (statsmodels)

`fleiss_kappa` expects a matrix with one row per item and one column per category, holding the **number of raters** who chose that category. `aggregate_raters` builds it from the raw label table.

In [3]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
raw = labels[["A1", "A2", "A3"]].to_numpy()
counts, cats = aggregate_raters(raw)          # counts: 20 x n_categories
kappa = fleiss_kappa(counts, method="fleiss")
raw_agreement = np.mean([len(set(r)) == 1 for r in raw])
print("categories:", list(cats))
print(f"Fleiss' kappa = {kappa:.3f}")
print(f"raw agreement (all three identical) = {raw_agreement:.2f}")

categories: ['Egyptian', 'Gulf', 'Levantine', 'MSA']
Fleiss' kappa = 0.687
raw agreement (all three identical) = 0.65


## 3. Pairwise Cohen's kappa (two-rater case) and a confusion matrix

In [4]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix
for a, b in [("A1","A2"), ("A1","A3"), ("A2","A3")]:
    print(f"Cohen's kappa {a} vs {b}: {cohen_kappa_score(labels[a], labels[b]):.3f}")
cm = confusion_matrix(labels["A1"], labels["A2"], labels=CLASSES)
pd.DataFrame(cm, index=[f"A1={c}" for c in CLASSES], columns=[f"A2={c}" for c in CLASSES])

Cohen's kappa A1 vs A2: 0.733
Cohen's kappa A1 vs A3: 0.797
Cohen's kappa A2 vs A3: 0.533


,A2=MSA,A2=Egyptian,A2=Gulf,A2=Levantine
A1=MSA,4,1,0,0
A1=Egyptian,0,5,0,0
A1=Gulf,0,0,4,2
A1=Levantine,0,0,1,3


## 4. What counts as usable agreement

There is no universal threshold (Section 7.5.3). The commonly quoted Landis and Koch bands are 0.41 to 0.60 "moderate", 0.61 to 0.80 "substantial", above 0.80 "almost perfect". For a four-class dialect task used as **training labels**, a Fleiss' kappa of about **0.6 or higher** on the pilot is usually taken as usable, provided the confusion matrix shows the disagreements are concentrated on genuinely ambiguous pairs (Gulf/Levantine, MSA/dialect mixing) rather than spread everywhere; for a **test set** the project should adjudicate every disagreement rather than rely on kappa at all. A kappa below about 0.4 means the guideline, not the annotators, needs fixing: usually the class definitions (what counts as MSA when a speaker mixes registers?) or the clip length (very short clips give too little evidence, Section 8.3).

## 5. What the one-page data card must contain

| Section | Content |
|---|---|
| Purpose | why the set was built; intended uses; uses it is not suitable for |
| Composition | number of clips, total duration, clips and minutes per dialect class, clip-length distribution |
| Source and collection | where the audio came from (programme, platform, recording setup), dates, consent and licence of the source, IRB or ethics status |
| Label taxonomy | the four classes, their definitions, how MSA-dialect mixing and unclear clips are handled |
| Annotation process | number of annotators, their backgrounds and native varieties, the guideline version, independent labelling, adjudication rule |
| Agreement | Fleiss' kappa (and pairwise Cohen's kappa) on the shared sample, sample size, the confusion matrix, how disagreements were resolved |
| Splits | how training, development and test were formed (speaker or programme disjoint), split sizes per class |
| Speaker and demographic metadata | which fields exist (gender, age band, region), how they were obtained (self-reported or annotator-assigned), counts per group |
| Known limitations | class imbalance, channel or programme shortcuts a classifier might learn, dialects not covered |
| Licence, access, maintenance | licence text, access conditions, version number, contact, how corrections are handled |